# Week 3 · Math for ML
Nights 15–21 · 7–13 December 2026

Each exercise is a code cell. Replace `raise NotImplementedError(...)` with your code and run the cell
with **Shift+Enter**: the checks underneath print **✅ Correct** when it works. Stuck for 20 minutes?
Compare with `solutions/week3_math_for_ml_solutions.ipynb`, then retype the answer yourself.

Tonight's reading and videos are in the study guide: [Week 3 tab](https://claude.ai/artifact/5MJxvaW9sNxjp7rjmPWuCu#3e1e9b7c-5e48).

In [ ]:
# Run this cell first (Shift+Enter). It finds your project folder and defines `check`,
# which prints "Correct" when an exercise works.
import math
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "data"


def _same(a, b, tol=1e-6):
    import math as _math
    import numbers as _numbers
    try:
        import numpy as _np
    except ImportError:
        _np = None
    if _np is not None and (isinstance(a, _np.ndarray) or isinstance(b, _np.ndarray)):
        a, b = _np.asarray(a), _np.asarray(b)
        if a.shape != b.shape:
            return False
        if a.dtype.kind in "biuf" and b.dtype.kind in "biuf":
            return bool(_np.allclose(a, b, rtol=1e-5, atol=tol))
        return bool((a == b).all())
    if isinstance(a, bool) or isinstance(b, bool):
        return a == b
    if isinstance(a, _numbers.Number) and isinstance(b, _numbers.Number):
        return _math.isclose(a, b, rel_tol=1e-5, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def check(label, actual, expected=True, tol=1e-6):
    """Run `actual` (usually a lambda) and print Correct if it equals `expected`."""
    try:
        got = actual() if callable(actual) else actual
    except NotImplementedError:
        print(f"⏳ Not done yet: {label}")
        return
    except Exception as error:  # show the problem without stopping the notebook
        print(f"❌ {label}: your code raised {type(error).__name__}: {error}")
        return
    if _same(got, expected, tol):
        print(f"✅ Correct: {label}")
    else:
        print(f"❌ {label}\n   expected: {expected!r}\n   got:      {got!r}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Ready")

## Night 15 · Mon 7 Dec: Vectors and the dot product
A vector is a list of numbers. The dot product measures how much two vectors point the same way; cosine
similarity is that idea scaled to the range −1 to 1, and it's how AI search finds text with a similar meaning.

In [ ]:
a = np.array([2, 1])
b = np.array([1, 3])
print("a + b =", a + b)
print("3 * a =", 3 * a)

In [ ]:
# 15.1 Length, dot product and cosine similarity
def vector_length(v):
    """Return the length of v. Hint: np.linalg.norm."""
    raise NotImplementedError("Your code here")


def dot_by_hand(a, b):
    """Multiply matching entries and add them up, with a loop or sum(). No NumPy."""
    raise NotImplementedError("Your code here")


def cosine_similarity(a, b):
    """Return dot(a, b) / (length(a) * length(b))."""
    raise NotImplementedError("Your code here")

check("length of [3, 4]", lambda: vector_length([3, 4]), 5.0)
check("dot_by_hand([1, 2, 3], [4, 5, 6])", lambda: dot_by_hand([1, 2, 3], [4, 5, 6]), 32)
check("same answer as np.dot", lambda: dot_by_hand([1, 2, 3], [4, 5, 6]) == np.dot([1, 2, 3], [4, 5, 6]), True)
check("at right angles: 0", lambda: cosine_similarity([1, 0], [0, 1]), 0.0)
check("same direction: 1", lambda: cosine_similarity([1, 1], [2, 2]), 1.0)
check("opposite: -1", lambda: cosine_similarity([1, 0], [-1, 0]), -1.0)

In [ ]:
# A made-up embedding table. Real embeddings have hundreds of numbers learned from text;
# these 4 stand for roughly: royalty, male, female, fruit.
embeddings = {
    "king": np.array([1.0, 0.6, 0.1, 0.0]),
    "queen": np.array([1.0, 0.1, 0.6, 0.0]),
    "man": np.array([0.1, 0.8, 0.1, 0.0]),
    "woman": np.array([0.1, 0.1, 0.8, 0.0]),
    "mango": np.array([0.0, 0.0, 0.0, 1.0]),
    "apple": np.array([0.05, 0.0, 0.0, 0.9]),
}

In [ ]:
# 15.2 Search by meaning
def closest_word(word, table):
    """Return the other word in `table` whose vector has the highest cosine similarity to `word`'s."""
    raise NotImplementedError("Your code here")


def analogy(a, b, c, table):
    """Return the word closest to vector(a) - vector(b) + vector(c), leaving out a, b and c themselves."""
    raise NotImplementedError("Your code here")

check("closest to king", lambda: closest_word("king", embeddings), "queen")
check("closest to mango", lambda: closest_word("mango", embeddings), "apple")
check("king - man + woman", lambda: analogy("king", "man", "woman", embeddings), "queen")

## Night 16 · Tue 8 Dec: Matrices as transformations
A matrix transforms vectors. Shapes (m, n) @ (n, p) give (m, p). A neural-network layer is a matrix multiply plus a bias.

In [ ]:
# 16.1 Rotation
def rotate(point, degrees):
    """Rotate a 2-D point counter-clockwise around (0, 0). Build R = [[cos θ, -sin θ], [sin θ, cos θ]]
    with θ = math.radians(degrees), and return R @ point."""
    raise NotImplementedError("Your code here")

check("rotate (1, 0) by 90°", lambda: rotate([1, 0], 90), np.array([0.0, 1.0]))
check("rotate (2, 0) by 180°", lambda: rotate([2, 0], 180), np.array([-2.0, 0.0]))

In [ ]:
# 16.2 The shape rule, and matrix multiplication with plain loops
def result_shape(shape_a, shape_b):
    """Return the shape of A @ B as a tuple, or None when the inner sizes don't match."""
    raise NotImplementedError("Your code here")


def matmul_loops(A, B):
    """Multiply two matrices given as lists of lists, using three nested loops. No NumPy."""
    raise NotImplementedError("Your code here")

check("(2, 3) @ (3, 4)", lambda: result_shape((2, 3), (3, 4)), (2, 4))
check("(2, 3) @ (2, 3) doesn't fit", lambda: result_shape((2, 3), (2, 3)), None)
check("2x2 by hand", lambda: matmul_loops([[1, 2], [3, 4]], [[5, 6], [7, 8]]), [[19, 22], [43, 50]])
check(
    "2x3 @ 3x2 matches NumPy",
    lambda: matmul_loops([[1, 2, 3], [4, 5, 6]], [[7, 8], [9, 10], [11, 12]]),
    (np.array([[1, 2, 3], [4, 5, 6]]) @ np.array([[7, 8], [9, 10], [11, 12]])).tolist(),
)

In [ ]:
# 16.3 One neural-network layer on a batch of 3 examples with 2 inputs each
X = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
W = np.array([[0.5, -1.0, 2.0], [1.0, 0.0, -1.0]])  # 2 inputs -> 3 outputs
b = np.array([0.1, 0.2, 0.3])


def dense_layer(X, W, b):
    """Return X @ W + b: one row of outputs per example."""
    raise NotImplementedError("Your code here")


def relu(z):
    """Replace negative numbers with 0. Hint: np.maximum."""
    raise NotImplementedError("Your code here")

check("dense_layer", lambda: dense_layer(X, W, b), np.array([[2.6, -0.8, 0.3], [5.6, -2.8, 2.3], [8.6, -4.8, 4.3]]))
check("relu(dense_layer)", lambda: relu(dense_layer(X, W, b)), np.array([[2.6, 0.0, 0.3], [5.6, 0.0, 2.3], [8.6, 0.0, 4.3]]))

## Night 17 · Wed 9 Dec: Derivatives, or how fast things change
A derivative is a slope: how much the output moves when the input moves a little. At a minimum the slope is zero.

In [ ]:
# 17.1 A numerical derivative
def numerical_derivative(f, x, h=1e-5):
    """Approximate f'(x) with the central difference (f(x + h) - f(x - h)) / (2 * h)."""
    raise NotImplementedError("Your code here")

check("slope of x² at x = 3 is 6", lambda: numerical_derivative(lambda x: x ** 2, 3.0), 6.0, tol=1e-4)
check("slope of sin at 0 is 1", lambda: numerical_derivative(np.sin, 0.0), 1.0, tol=1e-4)

In [ ]:
# 17.2 Plot f(x) = x² and its tangent line at x = 1: y = f(1) + slope * (x - 1)
f = lambda x: x ** 2
xs = np.linspace(-2, 3, 100)
fig, ax = plt.subplots(figsize=(6, 4))
# TODO: plot f(xs), work out the slope at 1 with numerical_derivative, plot the tangent as a dashed line
plt.show()

In [ ]:
# 17.3 Walk downhill using only the sign of the slope
def lowest_point(f, start=5.0, step=0.01, max_steps=10_000):
    """Start at x = start. While the slope at x isn't close to 0 (abs(slope) < 1e-3),
    move x by `step` against the slope's sign. Return x rounded to 2 decimals."""
    raise NotImplementedError("Your code here")

check("the lowest point of (x - 2)² + 1 is at x = 2", lambda: lowest_point(lambda x: (x - 2) ** 2 + 1), 2.0)

## Night 18 · Thu 10 Dec: Gradients and the chain rule
The gradient lists every partial derivative and points uphill. The chain rule multiplies slopes along a chain of
steps; backpropagation is the chain rule run backwards through a network.

In [ ]:
# 18.1 Partial derivatives: nudge one input at a time
def partial_derivatives(f, x, y, h=1e-5):
    """Return (df/dx, df/dy) for f(x, y), using central differences."""
    raise NotImplementedError("Your code here")


# 18.2 The gradient of a function of a whole vector
def gradient(f, point, h=1e-5):
    """Return an array with the partial derivative of f along each coordinate of `point`."""
    raise NotImplementedError("Your code here")

check("x² + 3xy at (1, 2)", lambda: partial_derivatives(lambda x, y: x ** 2 + 3 * x * y, 1.0, 2.0), (8.0, 3.0), tol=1e-4)
check("gradient of x² + y² at (1, 2)", lambda: gradient(lambda p: p[0] ** 2 + p[1] ** 2, [1.0, 2.0]), np.array([2.0, 4.0]), tol=1e-4)

In [ ]:
# 18.3 The chain rule
def chain_rule_derivative(x):
    """For g(x) = (3x + 1)², return g'(x) using the chain rule: 2(3x + 1) · 3."""
    raise NotImplementedError("Your code here")

check("chain rule at x = 2", lambda: chain_rule_derivative(2.0), 42.0)
check("matches the numerical slope", lambda: round(numerical_derivative(lambda x: (3 * x + 1) ** 2, 2.0), 3), 42.0)

In [ ]:
# 18.4 One step uphill against one step downhill
try:
    f = lambda p: p[0] ** 2 + p[1] ** 2
    p = np.array([1.0, 2.0])
    g = gradient(f, p)
    print("gradient:          ", g)
    print("f at p:            ", f(p))
    print("one step uphill:   ", f(p + 0.1 * g))
    print("one step downhill: ", f(p - 0.1 * g))
except NotImplementedError:
    print("Finish gradient() above first.")

## Night 19 · Fri 11 Dec: Gradient descent, or how models learn
Loss measures how wrong the model is. The gradient says which way to change each weight. A learning rate that's
too big makes training blow up.

In [ ]:
# 19.1 Gradient descent in one dimension
def minimize(f, df, x0, lr=0.1, steps=100):
    """Start at x0 and repeat x = x - lr * df(x) for `steps` steps. Return the final x."""
    raise NotImplementedError("Your code here")

check("minimum of (x - 3)² is at 3", lambda: minimize(lambda x: (x - 3) ** 2, lambda x: 2 * (x - 3), x0=0.0), 3.0, tol=1e-3)

In [ ]:
# 19.2 Try three learning rates on (x - 3)²: too small, good, too big
try:
    for lr in [0.001, 0.1, 1.1]:
        x_final = minimize(lambda x: (x - 3) ** 2, lambda x: 2 * (x - 3), x0=0.0, lr=lr, steps=50)
        print(f"lr = {lr}: x after 50 steps = {x_final:,.4f}")
except NotImplementedError:
    print("Finish minimize() above first.")

In [ ]:
# Noisy points around the line y = 2.5x + 4. Can gradient descent find 2.5 and 4?
rng = np.random.default_rng(0)
x_toy = rng.uniform(0, 10, 100)
y_toy = 2.5 * x_toy + 4 + rng.normal(0, 1, 100)

In [ ]:
# 19.3 Fit a line from scratch
def fit_line(x, y, lr=0.01, epochs=2000):
    """Fit y = w * x + b by gradient descent on the mean squared error, starting from w = b = 0.
    Each epoch: pred = w * x + b, error = y - pred, record (error ** 2).mean() in `losses`, then
    w -= lr * (-2 * (x * error).mean()) and b -= lr * (-2 * error.mean()).
    Return (w, b, losses)."""
    raise NotImplementedError("Your code here")

check("w matches np.polyfit", lambda: abs(fit_line(x_toy, y_toy)[0] - np.polyfit(x_toy, y_toy, 1)[0]) < 0.05, True)
check("b matches np.polyfit", lambda: abs(fit_line(x_toy, y_toy)[1] - np.polyfit(x_toy, y_toy, 1)[1]) < 0.05, True)
check("the loss went down", lambda: fit_line(x_toy, y_toy)[2][-1] < fit_line(x_toy, y_toy)[2][0], True)

In [ ]:
# 19.4 Plot the loss curve (log scale on the y axis shows the whole fall)
fig, ax = plt.subplots(figsize=(6, 4))
# TODO: w, b, losses = fit_line(x_toy, y_toy); ax.plot(losses); ax.set_yscale("log"); add labels
plt.show()

## Night 20 · Sat 12 Dec: Probability and statistics basics
A distribution is the shape of your data. Standard deviation is the typical distance from the mean. A classifier
estimates a conditional probability, like P(survived | female).

In [ ]:
# 20.1 Simulate, then summarize by hand
def coin_flips(n, seed=0):
    """Flip a fair coin n times with rng = np.random.default_rng(seed) and rng.integers(0, 2, size=n).
    Return the share of heads (1s)."""
    raise NotImplementedError("Your code here")


def mean_var_std(values):
    """Return (mean, variance, standard deviation) computed by hand with sum() and len(), no NumPy.
    Use the population variance: the average squared distance from the mean."""
    raise NotImplementedError("Your code here")


def z_scores(values):
    """Return (x - mean) / std for each value, as a NumPy array."""
    raise NotImplementedError("Your code here")

data = [2, 4, 4, 4, 5, 5, 7, 9]
check("100,000 flips land close to half heads", lambda: abs(coin_flips(100_000) - 0.5) < 0.01, True)
check("mean, variance, std", lambda: mean_var_std(data), (5.0, 4.0, 2.0))
check("z_scores", lambda: z_scores(data), np.array([-1.5, -0.5, -0.5, -0.5, 0.0, 0.0, 1.0, 2.0]))

In [ ]:
# 20.2 The normal distribution: 10,000 made-up heights with mean 170 cm and standard deviation 8 cm
rng = np.random.default_rng(1)
heights = rng.normal(170, 8, 10_000)
fig, ax = plt.subplots(figsize=(6, 4))
# TODO: histogram of heights with 50 bins; draw dashed vertical lines at the mean and at mean ± 1 std (ax.axvline)
plt.show()

In [ ]:
titanic = pd.read_csv(DATA / "titanic.csv")

In [ ]:
# 20.3 Conditional probability
def conditional_survival(df, sex):
    """Return P(survived | sex) from the Titanic data, as a fraction between 0 and 1."""
    raise NotImplementedError("Your code here")

check("P(survived | female)", lambda: round(conditional_survival(titanic, "female"), 3), 0.742)
check("P(survived | male)", lambda: round(conditional_survival(titanic, "male"), 3), 0.189)

## Night 21 · Sun 13 Dec: Review and project: linear regression from scratch
Predict a penguin's body mass from its flipper length with your own gradient descent from Night 19.
Standardize the input first, so one learning rate suits both w and b.

In [ ]:
penguins = pd.read_csv(DATA / "penguins.csv").dropna(subset=["flipper_length_mm", "body_mass_g"])
flipper = penguins["flipper_length_mm"].to_numpy(dtype=float)
mass = penguins["body_mass_g"].to_numpy(dtype=float)
print(len(flipper), "penguins")

In [ ]:
# 21.1 Helpers
def standardize(values):
    """Return (values - mean) / std."""
    raise NotImplementedError("Your code here")


def mae(y_true, y_pred):
    """Mean absolute error: the average size of the mistakes."""
    raise NotImplementedError("Your code here")


def predict_mass(flipper_mm, w, b, mean, std):
    """Predict body mass (g) for a flipper length in mm, with a model trained on standardized input."""
    raise NotImplementedError("Your code here")

check("standardize gives mean 0, std 1", lambda: [round(float(standardize(flipper).mean()), 6), round(float(standardize(flipper).std()), 6)], [0.0, 1.0])
check("mae", lambda: mae([1, 2, 3], [2, 2, 2]), 2 / 3)
check("predict_mass", lambda: predict_mass(200, 697.64, 4201.75, 200.915, 14.041), 4156.3, tol=0.1)

In [ ]:
# 21.2 Train, compare with NumPy's answer, and measure the error
try:
    x = standardize(flipper)
    w, b, losses = fit_line(x, mass, lr=0.1, epochs=500)
    w_np, b_np = np.polyfit(x, mass, 1)
    print(f"gradient descent: w = {w:.2f}, b = {b:.2f}")
    print(f"np.polyfit:       w = {w_np:.2f}, b = {b_np:.2f}")
    print(f"mean absolute error: {mae(mass, w * x + b):.0f} g")
    guess = predict_mass(200, w, b, flipper.mean(), flipper.std())
    print(f"a penguin with 200 mm flippers: about {guess:,.0f} g")
except NotImplementedError:
    print("Finish standardize(), mae(), predict_mass() and fit_line() (Night 19) first.")

In [ ]:
check("your w matches np.polyfit", lambda: abs(w - w_np) < 1, True)
check("mean absolute error is about 313 g", lambda: round(mae(mass, w * x + b)), 313)

In [ ]:
# 21.3 Plot the data with your line, and the loss curve
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
# TODO: left: scatter flipper vs mass, plus your line (sort the x values first);
#       right: the losses from fit_line, log scale
plt.show()

You just built the loop inside every ML model: predict, measure the loss, follow the gradient, repeat.
Push your work, then tick off Week 3 in the study guide.